# 23 Tabel dan Gambar


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json
import os
import sys
import hashlib

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next(
    (p.resolve() for p in _candidates
     if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()),
    None
)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')

LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('23_Tabel_dan_Gambar')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

for _name in ['experiment_lib', 'care_km']:
    sys.modules.pop(_name, None)
import experiment_lib as el
import care_km as ck

METHODS = list(el.METHODS)
CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

print('PAPER_ROOT:', PAPER_ROOT)
print('DATA_DIR  :', DATA_DIR)
print('METHODS   :', METHODS)
print('CARE config:', CARE_CONFIG)


import matplotlib.pyplot as plt

FIG_DIR = artifact_path('FIGURES','23_Tabel_dan_Gambar')
FIG_DIR.mkdir(parents=True, exist_ok=True)

required = {
    'lms_full': artifact_path('04_full_dataset_summary.csv','23_Tabel_dan_Gambar'),
    'refinement_trajectory': artifact_path('05_refinement_trajectory_summary.csv','23_Tabel_dan_Gambar'),
    'lms_robustness': artifact_path('06_robustness_summary.csv','23_Tabel_dan_Gambar'),
    'lms_robustness_raw': artifact_path('06_robustness_raw.csv','23_Tabel_dan_Gambar'),
    'lms_stats': artifact_path('07_wilcoxon_holm.csv','23_Tabel_dan_Gambar'),
    'fslsm_mapping': artifact_path('08_cluster_to_fslsm_mapping.csv','23_Tabel_dan_Gambar'),
    'gen_audit': artifact_path('09_generalization_dataset_audit.csv','23_Tabel_dan_Gambar'),
    'gen_full': artifact_path('10_cross_dataset_full.csv','23_Tabel_dan_Gambar'),
    'gen_rank': artifact_path('10_average_rank.csv','23_Tabel_dan_Gambar'),
    'gen_stats': artifact_path('11_cross_dataset_pairwise_wilcoxon_holm.csv','23_Tabel_dan_Gambar'),
    'friedman': artifact_path('11_friedman_kendall.csv','23_Tabel_dan_Gambar'),
    'runtime': artifact_path('12_resample_runtime_scalability.csv','23_Tabel_dan_Gambar'),
    'stability': artifact_path('12_stability_failure_method_summary.csv','23_Tabel_dan_Gambar'),
}
missing = [str(p) for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Output prasyarat belum lengkap:\n' + '\n'.join(missing)
    )

frames = {k: pd.read_csv(v) for k, v in required.items()}
print('Generalization datasets:', frames['gen_audit']['dataset'].nunique())


tables = {}

tables['13_table_main_lms_classification.csv'] = pd.read_csv(artifact_path('70_30_main_lms_classifier_metrics.csv'))

tables['13_table_01_lms_full_data.csv'] = frames['lms_full'].copy()
tables['13_table_02_refinement_trajectory.csv'] = frames['refinement_trajectory'].copy()
tables['13_table_03_lms_robustness.csv'] = frames['lms_robustness'].copy()
tables['13_table_04_lms_statistics.csv'] = frames['lms_stats'].copy()

fslsm = frames['fslsm_mapping'].copy().sort_values('cluster_id')

required_fslsm_cols = [
    'active_reflective_score', 'sensing_intuitive_score',
    'visual_verbal_score', 'sequential_global_score',
    'active_reflective_category', 'sensing_intuitive_category',
    'visual_verbal_category', 'sequential_global_category',
    'active_reflective_code', 'sensing_intuitive_code',
    'visual_verbal_code', 'sequential_global_code',
    'preference_signature',
    'mean_behavioral_preference_strength',
    'boundary_dimension', 'nearest_alternative_ls',
    'behavioral_profile_variant',
]
missing_fslsm = [
    c for c in required_fslsm_cols if c not in fslsm.columns
]
if missing_fslsm:
    raise RuntimeError(
        'Output FSLSM belum memakai skala 11-equivalent: '
        + ', '.join(missing_fslsm)
    )

mapping_cols = [
    'cluster_id', 'cluster_name', 'cluster_size',
    'ls_class', 'semantic_label',
    'active_reflective', 'sensing_intuitive',
    'visual_verbal', 'sequential_global',
    'active_reflective_score', 'sensing_intuitive_score',
    'visual_verbal_score', 'sequential_global_score',
    'active_reflective_category', 'sensing_intuitive_category',
    'visual_verbal_category', 'sequential_global_category',
    'active_reflective_code', 'sensing_intuitive_code',
    'visual_verbal_code', 'sequential_global_code',
    'preference_signature',
    'mean_behavioral_preference_strength',
    'boundary_dimension', 'nearest_alternative_ls',
    'behavioral_profile_variant', 'mapping_status',
]
tables['13_table_05_fslsm_mapping.csv'] = fslsm[mapping_cols].copy()

tables['13_table_05b_fslsm_strength_profiles.csv'] = (
    fslsm[[
        'cluster_name', 'cluster_size', 'ls_class', 'semantic_label',
        'active_reflective_code', 'active_reflective_category',
        'sensing_intuitive_code', 'sensing_intuitive_category',
        'visual_verbal_code', 'visual_verbal_category',
        'sequential_global_code', 'sequential_global_category',
        'mean_behavioral_preference_strength',
        'boundary_dimension', 'nearest_alternative_ls',
        'preference_signature',
    ]]
    .rename(columns={
        'cluster_name': 'empirical_cluster',
        'cluster_size': 'learners',
        'ls_class': 'base_ls_class',
        'semantic_label': 'base_fslsm_profile',
        'active_reflective_code': 'A_R_score',
        'active_reflective_category': 'A_R_interpretation',
        'sensing_intuitive_code': 'Sens_Int_score',
        'sensing_intuitive_category': 'Sens_Int_interpretation',
        'visual_verbal_code': 'Vis_Ver_score',
        'visual_verbal_category': 'Vis_Ver_interpretation',
        'sequential_global_code': 'Seq_Gl_score',
        'sequential_global_category': 'Seq_Gl_interpretation',
        'mean_behavioral_preference_strength': 'mean_strength',
        'boundary_dimension': 'boundary_dimension',
        'nearest_alternative_ls': 'nearest_alternative_ls',
        'preference_signature': 'preference_signature',
    })
)

tables['13_table_06_generalization_datasets.csv'] = frames['gen_audit'].copy()
tables['13_table_07_generalization_average_rank.csv'] = frames['gen_rank'].copy()
tables['13_table_08_cross_dataset_friedman_kendall.csv'] = frames['friedman'].copy()
tables['13_table_09_cross_dataset_pairwise.csv'] = frames['gen_stats'].copy()
tables['13_table_10_runtime_scalability.csv'] = frames['runtime'].copy()
tables['13_table_11_stability_failure.csv'] = frames['stability'].copy()
tables['13_table_12_generalization_care_full.csv'] = (
    frames['gen_full'][frames['gen_full']['method'].eq('CARE-KM')].copy()
)

for name, frame in tables.items():
    frame.to_csv(artifact_path(name,'23_Tabel_dan_Gambar'), index=False)

print('Tables generated:', len(tables))
for name in tables:
    print(' -', name)

# Figure 1 — LMS full-data: DBI.
d = frames['lms_full'].set_index('method').reindex(METHODS).reset_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(d['method'], d['dbi'])
ax.set_ylabel('DBI (lower is better)')
ax.set_xlabel('Initialization method')
ax.set_title('LMS full-data clustering quality — DBI')
ax.tick_params(axis='x', rotation=35)
fig.tight_layout()
fig.savefig(FIG_DIR / '13_fig_01_lms_full_dbi.png', dpi=200)
plt.show()


# Figure 2 — LMS full-data: Silhouette Index.
d = frames['lms_full'].set_index('method').reindex(METHODS).reset_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(d['method'], d['si'])
ax.set_ylabel('Silhouette Index (higher is better)')
ax.set_xlabel('Initialization method')
ax.set_title('LMS full-data clustering quality — Silhouette Index')
ax.tick_params(axis='x', rotation=35)
fig.tight_layout()
fig.savefig(FIG_DIR / '13_fig_02_lms_full_si.png', dpi=200)
plt.show()


# Figure 3 — Cross-dataset average rank for DBI (resampling mean).
rank = frames['gen_rank']
d = rank[(rank['source']=='resample_mean') & (rank['metric']=='dbi')].copy()
d = d.set_index('method').reindex(METHODS).reset_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(d['method'], d['average_rank'])
ax.set_ylabel('Average rank (lower is better)')
ax.set_xlabel('Initialization method')
ax.set_title('Cross-dataset average rank — DBI')
ax.tick_params(axis='x', rotation=35)
fig.tight_layout()
fig.savefig(FIG_DIR / '13_fig_03_generalization_rank_dbi.png', dpi=200)
plt.show()


# Figure 4 — Cross-dataset average rank for SI (resampling mean).
rank = frames['gen_rank']
d = rank[(rank['source']=='resample_mean') & (rank['metric']=='si')].copy()
d = d.set_index('method').reindex(METHODS).reset_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(d['method'], d['average_rank'])
ax.set_ylabel('Average rank (lower is better)')
ax.set_xlabel('Initialization method')
ax.set_title('Cross-dataset average rank — Silhouette Index')
ax.tick_params(axis='x', rotation=35)
fig.tight_layout()
fig.savefig(FIG_DIR / '13_fig_04_generalization_rank_si.png', dpi=200)
plt.show()


# Figure 5 — CARE-KM resampling runtime versus dataset size.
runtime = frames['runtime']
d = runtime[runtime['method']=='CARE-KM'].copy().sort_values('rows')
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(d['rows'], d['total_sec_mean'])
ax.set_xlabel('Dataset rows')
ax.set_ylabel('Mean total runtime (s)')
ax.set_title('CARE-KM runtime across generalization datasets')
fig.tight_layout()
fig.savefig(FIG_DIR / '13_fig_05_care_runtime_scaling.png', dpi=200)
plt.show()


# Figure 6 — paired DBI improvement across LMS resamples.
raw = frames['lms_robustness_raw'].copy()
baselines = [m for m in METHODS if m != 'CARE-KM']

pivot = raw.pivot(index='rep', columns='method', values='dbi').sort_index()
missing_methods = [m for m in METHODS if m not in pivot.columns]
if missing_methods:
    raise RuntimeError('Metode tidak lengkap pada robustness DBI: ' + ', '.join(missing_methods))

dbi_gain = [
    (pivot[m] - pivot['CARE-KM']).dropna().to_numpy()
    for m in baselines
]
if any(len(v) != 30 for v in dbi_gain):
    raise RuntimeError('Paired DBI robustness harus berisi 30 resample per baseline.')

fig, ax = plt.subplots(figsize=(8.4, 4.8))
ax.boxplot(dbi_gain, tick_labels=baselines, showmeans=True)
rng = np.random.default_rng(1306)
for x, values in enumerate(dbi_gain, start=1):
    jitter = rng.uniform(-0.07, 0.07, len(values))
    ax.scatter(np.full(len(values), x) + jitter, values, s=14, alpha=0.55)
ax.axhline(0, linewidth=0.9, linestyle='--')
ax.set_ylabel('Paired DBI improvement')
ax.set_xlabel('Reference initializer')
ax.set_title('CARE-KM paired DBI improvement across 30 LMS resamples')
ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(FIG_DIR / '13_fig_06_lms_robustness_dbi_paired.png', dpi=200, bbox_inches='tight')
plt.show()


# Figure 7 — paired SI improvement across LMS resamples.
raw = frames['lms_robustness_raw'].copy()
baselines = [m for m in METHODS if m != 'CARE-KM']

pivot = raw.pivot(index='rep', columns='method', values='si').sort_index()
missing_methods = [m for m in METHODS if m not in pivot.columns]
if missing_methods:
    raise RuntimeError('Metode tidak lengkap pada robustness SI: ' + ', '.join(missing_methods))

si_gain = [
    (pivot['CARE-KM'] - pivot[m]).dropna().to_numpy()
    for m in baselines
]
if any(len(v) != 30 for v in si_gain):
    raise RuntimeError('Paired SI robustness harus berisi 30 resample per baseline.')

fig, ax = plt.subplots(figsize=(8.4, 4.8))
ax.boxplot(si_gain, tick_labels=baselines, showmeans=True)
rng = np.random.default_rng(1307)
for x, values in enumerate(si_gain, start=1):
    jitter = rng.uniform(-0.07, 0.07, len(values))
    ax.scatter(np.full(len(values), x) + jitter, values, s=14, alpha=0.55)
ax.axhline(0, linewidth=0.9, linestyle='--')
ax.set_ylabel('Paired SI improvement')
ax.set_xlabel('Reference initializer')
ax.set_title('CARE-KM paired SI improvement across 30 LMS resamples')
ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(FIG_DIR / '13_fig_07_lms_robustness_si_paired.png', dpi=200, bbox_inches='tight')
plt.show()


# Figure 8 — FSLSM behavioral preference signatures.
fs = frames['fslsm_mapping'].copy().sort_values('cluster_id')
score_cols = [
    'active_reflective_score',
    'sensing_intuitive_score',
    'visual_verbal_score',
    'sequential_global_score',
]
code_cols = [
    'active_reflective_code',
    'sensing_intuitive_code',
    'visual_verbal_code',
    'sequential_global_code',
]

values = fs[score_cols].apply(pd.to_numeric, errors='coerce')
if len(fs) != 16 or values.isna().any().any():
    raise RuntimeError('FSLSM mapping harus berisi 16 cluster dengan empat skor yang lengkap.')

allowed = {1, 3, 5, 7, 9, 11}
if not set(values.abs().stack().astype(int).unique()).issubset(allowed):
    raise RuntimeError('Skor FSLSM tidak sesuai skala 11-equivalent.')

labels = (
    fs['cluster_name'].astype(str)
    + '  (' + fs['ls_class'].astype(str) + ')'
).tolist()
annotations = fs[code_cols].astype(str).to_numpy()

fig, ax = plt.subplots(figsize=(7.2, 7.4))
im = ax.imshow(values.to_numpy(), vmin=-11, vmax=11, cmap='RdBu_r', aspect='auto')

ax.set_xticks(range(4), ['A/R', 'Sens./Int.', 'Vis./Ver.', 'Seq./Gl.'])
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel('FSLSM dimension')
ax.set_ylabel('Empirical cluster (base LS)')
ax.set_title('FSLSM behavioral preference signatures of LMS clusters')

for i in range(len(labels)):
    for j in range(4):
        ax.text(j, i, annotations[i, j], ha='center', va='center', fontsize=8)

cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('Signed 11-equivalent preference score')
cbar.set_ticks([-11, -9, -7, -5, -3, -1, 1, 3, 5, 7, 9, 11])

fig.text(
    0.5, 0.015,
    'Positive poles: Act., Sens., Vis., Seq.   |   Negative poles: Ref., Int., Ver., Gl.',
    ha='center', fontsize=8.5
)
fig.tight_layout(rect=[0, 0.035, 1, 1])
fig.savefig(
    FIG_DIR / '13_fig_08_fslsm_preference_signature_heatmap.png',
    dpi=220, bbox_inches='tight'
)
plt.show()


# Reproducibility manifest for manuscript outputs.
artifact_paths = sorted(
    [artifact_path(name,'23_Tabel_dan_Gambar') for name in tables]
    + list(FIG_DIR.glob('13_fig_*.png'))
)

manifest = {
    'table_count': len(tables),
    'figure_count': len(list(FIG_DIR.glob('13_fig_*.png'))),
    'generalization_dataset_count': int(frames['gen_audit']['dataset'].nunique()),
    'generalization_source': 'REVISED/DATA',
    'artifacts': {
        str(p.relative_to(PAPER_ROOT)): sha256_file(p)
        for p in artifact_paths if p.exists()
    },
    'note': 'No clustering is executed in notebook 13.',
}
(artifact_path('13_paper_artifact_manifest.json','23_Tabel_dan_Gambar')).write_text(
    json.dumps(manifest, indent=2), encoding='utf-8'
)
print(json.dumps(manifest, indent=2))
